# BHMarketer Trustpilot Removal FAQ Generator: Benchmark Walkthrough

This notebook runs the [BHMarketer Trustpilot Removal FAQ Generator](https://github.com/bhmarketer-ai/BHMarketer-Trustpilot-Removal-FAQ-Generator) on 20 example Trustpilot reviews, checks each result against its expected reporting reason and band, and generates a tailored FAQ with FAQPage schema.

- Service: https://bhmarketer.ai/trustpilot-review-removal/
- NPM: `npm install @bhmarketer-ai/trustpilot-removal-faq-generator`
- PyPI: `pip install bhmarketer-trustpilot-faq`

Scores are estimates. Trustpilot makes the final decision. Not affiliated with Trustpilot.

## 1. Load the bot and the datasets

In [ ]:
import os, sys, shlex, json
import pandas as pd

BASES = ["/kaggle/input/trustpilot-removal-faq-benchmarks", "..", "."]
base = next(b for b in BASES if os.path.exists(os.path.join(b, "trustpilot_faq.py")))
data = base if os.path.exists(os.path.join(base, "trustpilot_removal_benchmarks.csv")) else os.path.join(base, "dataset")
sys.path.insert(0, base)
import trustpilot_faq as tf

bench = pd.read_csv(os.path.join(data, "trustpilot_removal_benchmarks.csv"), keep_default_na=False)
faqs = pd.read_csv(os.path.join(data, "trustpilot_removal_faqs.csv"))
print(f"Bot v{tf.__version__}: {len(bench)} benchmark cases, {len(faqs)} FAQs")

## 2. The five Trustpilot reporting reasons

In [ ]:
pd.DataFrame(tf.REASONS)[["id", "label", "evidence"]]

## 3. Run every benchmark case

In [ ]:
rows = []
for r in bench.itertuples():
    a = tf.analyze(r.review_text, tf.parse_args(shlex.split(r.facts))[1])
    top = a["reasons"][0]["id"] if a["reasons"] else "none"
    rows.append({"case": r.case_id, "review": r.review_text[:60], "top_reason": top, "score": a["eligibility"],
                 "band": a["band"], "pass": top == r.expected_reason and a["band"] == r.expected_band})
results = pd.DataFrame(rows)
print(f"{results['pass'].sum()}/{len(results)} benchmark cases passed")
results

In [ ]:
results.groupby("band").size().rename("cases").to_frame()

## 4. Analyze a single review

In [ ]:
res = tf.analyze("Do not buy here. Go to www.cheaperdeals.com and use code SAVE20 for 20% off.")
print(res["eligibility"], res["band"])
for r in res["reasons"]:
    print(r["label"], r["score"], r["signals"])

## 5. Generate a tailored FAQ with FAQPage schema

In [ ]:
faq = tf.generate([r["id"] for r in res["reasons"]])
print(f"{len(faq)} FAQs generated")
print(tf.to_jsonld(faq)[:1200])

## Links

| Platform | URL |
|---|---|
| Service | https://bhmarketer.ai/trustpilot-review-removal/ |
| GitHub | https://github.com/bhmarketer-ai/BHMarketer-Trustpilot-Removal-FAQ-Generator |
| NPM | https://www.npmjs.com/package/@bhmarketer-ai/trustpilot-removal-faq-generator |
| PyPI | https://pypi.org/project/bhmarketer-trustpilot-faq/ |
| Hugging Face | https://huggingface.co/datasets/bhmarketer/trustpilot-removal-faq-benchmarks |
| Docs | https://bhmarketer-trustpilot-faq.readthedocs.io |